# 1. 概述
记忆是一种记住之前互动信息的系统。随着Agent处理涉及大量用户交互的复杂任务，记忆变得至关重要！

大多数的大模型应用程序都会有一个 会话接口 ，允许我们进行 多轮的对话 ，并有一定的上下文记忆能力。

但实际上，大模型本身是“无状态”的， 不会记忆 任何上下文的。即每次调用 agent.invoke() 都是全新的开始，不记得之前的对话。

## 1.1 演示：Agent 默认不会记住上一次调用

下面用同一个 `agent` 连续调用两次 `invoke`：

* 第一次：`messages1` 里除了新问题，还**手动**放进了前一轮的对话（“你好，我叫小明” / “很高兴认识你，小明”）。
* 第二次：只发送一个新问题“我叫什么名字？”。

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="gpt-6-luna",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

In [4]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage

messages1 = [
    HumanMessage("你好，我叫小明"),
    AIMessage("很高兴认识你，小明"),
    HumanMessage("你用一句话介绍下你自己"),
]

agent = create_agent(
    model=model,
    tools=[],
)

response1 = agent.invoke({"messages" : messages1})

for msg in response1["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我叫小明
================================== Ai Message ==================================

很高兴认识你，小明
================================ Human Message =================================

你用一句话介绍下你自己
================================== Ai Message ==================================

我是一个由人工智能驱动的助手，可以帮助你解答问题、整理信息、创作内容和解决各种任务。


In [5]:
messages2 = [HumanMessage("我叫什么名字？")]

response2 = agent.invoke({"messages" : messages2})

for msg in response2["messages"]:
    msg.pretty_print()

================================ Human Message =================================

我叫什么名字？
================================== Ai Message ==================================

我还不知道你的名字。


### 1.1.1 为什么第二次就“忘了”

对比两次调用发给模型的内容：

| | 第一次调用 | 第二次调用 |
|---|---|---|
| 发给模型的消息 | 3 条：前一轮的问答（手动写入）+ 新问题 | 1 条：新问题 |
| 模型能看到“小明”吗 | 能，就在本次请求里 | 不能 |

* 第二次调用返回的 `messages` 只有 2 条，第一次调用的消息没有被带过来。`agent` 对象在两次 `invoke` 之间不保存任何东西，大模型的接口也不保存。
* 第一次调用时，模型之所以“知道”用户叫小明，是因为我们**手动**把前一轮对话写进了 `messages1`。这就是最原始的“记忆”：由程序保存历史，下一次请求时一起发给模型。

![为什么需要记忆](assets/01-why-memory.svg)

下面按图中 B 的做法，把第一次调用返回的全部消息接在新问题前面，再问一次（会请求一次模型）：

In [6]:
# 第一次调用返回的 4 条消息 + 新问题，一起发给模型
messages3 = response1["messages"] + [HumanMessage("我叫什么名字？")]

response3 = agent.invoke({"messages": messages3})

for msg in response3["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我叫小明
================================== Ai Message ==================================

很高兴认识你，小明
================================ Human Message =================================

你用一句话介绍下你自己
================================== Ai Message ==================================

我是一个由人工智能驱动的助手，可以帮助你解答问题、整理信息、创作内容和解决各种任务。
================================ Human Message =================================

我叫什么名字？
================================== Ai Message ==================================

你叫小明。


这次模型答出了名字。由此可见：

1. 模型“记得”什么，只取决于**本次请求发给它的内容**，也就是上下文。
2. 记忆要靠模型之外的程序实现：保存历史（图中 ①），下次请求时取出来放进上下文（图中 ②）。
3. 手动拼接消息列表很快会遇到麻烦：每个用户、每个会话的消息都要自己保存；对话越长，每次发送的内容越多。LangChain 的记忆组件就是来解决这些问题的。第七章 01 笔记 2.5.5 用过的 `checkpointer`，就是让框架自动完成 ① 和 ②。

## 1.2 如何解决记忆问题
### 1.2.1 上下文工程
实现这个记忆功能，就需要 额外的模块 去保存我们和模型对话的上下文信息，然后在下一次请求时，把历史信息都输入给模型，让模型输出结果。

在 LangChain 中， 记忆(Memory) 就是专门负责“ 存储历史交互信息 ”的组件，核心作用是「 保存上下文 」和「 提供上下文 」，让LLM在每次响应时，都能“看到”之前的对话内容。

上下文工程(Context Engineering) 负责 “合理组织”这些记忆和任务信息 ，让LLM的响应更连贯、更贴合需求。这也是Agent能实现 复杂多轮交互 的核心基础。

#### 补充：记忆和上下文工程是什么关系

两者分工不同：

| | 记忆（Memory） | 上下文工程（Context Engineering） |
|---|---|---|
| 解决的问题 | 信息**存在哪、怎么取** | 每次调用模型时，**放哪些信息、以什么形式放** |
| 打个比方 | 仓库 | 每次从仓库里挑货、打包、发货 |

为什么不能把所有历史都原样发给模型？模型一次能接收的 token 数（上下文窗口）是有限的，对话越长，问题越多：

1. **超出上下文窗口**：消息总量超过窗口上限，请求直接报错。
2. **模型分心**：即使没超出，大量与当前问题无关的旧内容也会干扰模型，让它忽略关键信息。
3. **又贵又慢**：每次请求都要重新发送全部历史，token 越多，费用越高、响应越慢。

所以光“存”不够，还要决定“给什么”。和记忆相关的上下文工程主要做三件事：

| 操作 | 含义 | LangChain 中的做法（后面会学） |
|---|---|---|
| 写入 | 把对话、或从对话中提炼出的信息保存起来 | 短期：`checkpointer` 自动保存；长期：在工具或中间件中调用 `store.put` |
| 选取 | 只取出与当前任务相关的部分 | `store.get` 按 key 读取；`store.search` 按条件或语义检索 |
| 压缩 | 历史太长时把它变短 | 裁剪（只保留最近的消息）、删除（`RemoveMessage`）、摘要（第八章的 `SummarizationMiddleware`） |

> 官方文档中的上下文工程不只针对记忆，还包括动态调整提示词、工具、模型等，第八章的中间件就是实现手段。本章只关注其中与记忆相关的部分。官方说明：https://docs.langchain.com/oss/python/langchain/context-engineering

### 1.2.2 上下文类型及相关的API

LangChain的上下文工程是基于Agent讨论的，而上下文工程是构建在LangGraph之上的。

LangGraph 提供了多种管理上下文的方法，这些方法结合了可变性和生命周期维度：

| 上下文类型 | 存什么 | 可变性 | 生命周期 | 怎么传入 / 读取 | 和记忆的关系 |
|---|---|---|---|---|---|
| 静态运行时上下文<br>（Runtime Context） | 用户 id、权限、数据库连接等，调用前就确定的信息 | 静态：运行中不变 | 一次运行 | 创建 Agent 时用 `context_schema` 声明结构，调用时 `invoke(..., context=...)` 传入；中间件、工具中通过 `runtime.context` 读取 | 不是记忆，每次调用都要重新传入 |
| 动态运行时上下文<br>（State） | 消息列表 `messages`，以及自定义字段 | 动态：每个节点都可以更新 | 一次运行；配置 `checkpointer` 后按 `thread_id` 保存，同一 thread 的下一次运行接着用 | 中间件 hook 的 `state` 参数；工具中的 `runtime.state` | **短期记忆** |
| 动态跨会话上下文<br>（Store） | 用户资料、偏好、从对话中提炼出的事实 | 动态：随时可以读写 | 跨会话，长期保存 | 创建 Agent 时传入 `store=...`；中间件、工具中通过 `runtime.store` 读写 | **长期记忆** |

按“可变性 × 生命周期”两个维度排一下：

| | 一次运行 | 跨会话 |
|---|---|---|
| 静态 | Runtime Context | —（如写死在代码里的系统提示词，不在这里讨论） |
| 动态 | State（配合 checkpointer 成为短期记忆） | Store（长期记忆） |

下图把三类上下文放进一次模型调用里看：

![一次模型调用的上下文从哪里来](assets/02-context-sources.svg)

几点说明：

* 第八章 07 笔记 5.3.3 讲过 hook 的两个参数：`state` 就是这里的动态运行时上下文；`runtime` 里的 `context`、`store` 分别是静态运行时上下文和跨会话上下文。工具则通过 `ToolRuntime` 类型的参数拿到同样的 `state`、`context`、`store`（后面学）。
* 官方表格中 State 的生命周期是“一次运行”：一次 `invoke` 结束，state 也就结束了。配置 `checkpointer` 后，state 会按 `thread_id` 保存下来，下次用同一个 `thread_id` 调用时恢复，这才成为短期记忆。
* Runtime Context 本身不是记忆，但长期记忆通常离不开它：`user_id` 一般通过 context 传入，用来决定读写 Store 中的哪个命名空间。

官方说明：https://docs.langchain.com/oss/python/concepts/context

## 1.3 LangChain的记忆
### 1.3.1 记忆的分类
官方说明：https://docs.langchain.com/oss/python/concepts/memory

记忆分为短期记忆和长期记忆，对应不同的使用场景：
1. 短期记忆（Short-term memory、会话级记忆、thread-scoped memory）：作用范围是单个对话线程（Thread）内，一旦开启新对话（更换 `thread_id` ），新对话就看不到之前的记忆（旧对话的记忆并没有删除，换回原来的 `thread_id` 还能接着聊）。
2. 长期记忆（Long-term memory，跨会话级记忆 ）：在会话间存储用户特定或应用级数据， 并在 `会话线程间共享` 。它可以随时在任何线程中被调用。记忆的范围是任意自定义命名空间，而不仅仅是单一线程 ID。

注意: 短期记忆不一定存储在内存中，也可以持久化到磁盘。只要作用范围是线程即可。长期记忆同理

#### 1. 什么是线程（Thread）

线程就是**一次会话**，用 `thread_id` 区分。可以对照 ChatGPT、豆包这类聊天应用来理解：

* 左侧会话列表里的每一个会话就是一个 thread。点开某个会话可以接着聊，因为这个会话的历史消息被保存着，这是**短期记忆**。
* 新建一个会话，它看不到其他会话里聊过的内容。
* 但应用的“记忆”功能会记住你的名字、职业、偏好，在新会话里也知道，这是**长期记忆**。

![短期记忆 vs 长期记忆](assets/03-short-vs-long-term.svg)

#### 2. 短期记忆与长期记忆对比

| | 短期记忆 | 长期记忆 |
|---|---|---|
| 作用范围 | 单个 thread 内 | 跨 thread，任何会话都能读写 |
| 用什么区分 | `thread_id` | 命名空间 `namespace` + 键 `key`，例如 `("users", user_id)` + `"profile"` |
| 存什么 | 当前会话的消息列表，以及自定义的 state 字段 | 从对话中提炼出的信息：用户资料、偏好、事实、经验等，以 JSON 文档保存 |
| 存储组件 | `checkpointer`（检查点保存器） | `store` |
| `create_agent` 参数 | `checkpointer=InMemorySaver()` | `store=InMemoryStore()` |
| 怎么写入 | **自动**：每一步执行完，state 自动保存 | **手动**：自己在工具或中间件里调用 `store.put` |
| 怎么读取 | **自动**：用同一个 `thread_id` 再次调用时自动恢复 | **手动**：调用 `store.get` / `store.search`，再放进提示词或作为工具结果返回 |
| 主要难点 | 对话越来越长，需要裁剪、摘要 | 存什么、什么时候存、怎么检索到相关的内容 |
| 学习、测试用的实现 | `InMemorySaver`：存在内存里，程序结束就没了 | `InMemoryStore`：同左 |
| 生产环境的实现 | `PostgresSaver`、`SqliteSaver`、`RedisSaver` 等，需另装 `langgraph-checkpoint-postgres` 等包 | `PostgresStore`、`RedisStore` 等 |

> 一句话区分：**短期记忆记的是“这次聊了什么”，由框架自动管理；长期记忆记的是“关于这个用户、这件事我知道什么”，由你决定存什么、什么时候存。**

#### 3. 长期记忆存什么（了解）

官方文档借用心理学的说法，把长期记忆分为三种：

| 类型 | 存什么 | 人的例子 | Agent 的例子 |
|---|---|---|---|
| 语义记忆（Semantic） | 事实 | 在学校学到的知识 | 用户叫小明、喜欢简洁的回答 |
| 情景记忆（Episodic） | 经历 | 自己做过的事 | 过去成功完成某个任务的过程，常作为 few-shot 示例放进提示词 |
| 程序记忆（Procedural） | 做事的规则 | 骑车、打字这类技能 | 系统提示词；Agent 根据用户反馈改写自己的提示词 |

最常用的是语义记忆，组织方式有两种：

* **档案（Profile）**：每个用户一份 JSON 文档，有新信息就更新这份文档。读取方便，但每次更新都要把旧档案交给模型改写，信息多了容易出错。
* **集合（Collection）**：每条信息单独存一条记录，有新信息就新增一条。不容易漏记，但要处理重复、过时的记录，读取时也要先检索出相关的几条。

写入长期记忆的时机也有两种：

* **在主流程中写入（hot path）**：Agent 在对话中通过工具当场保存，下一轮就能用上；缺点是增加响应延迟，模型还要分心判断“该记什么”。
* **在后台写入（background）**：对话结束后或定时由单独的任务从历史中提炼记忆；不影响响应速度，但新记忆不能马上用上。

### 1.3.2 记忆的管理
在LangChain v0.x版本中，通过专用的xxxMemory类管理记忆。

在LangChain v1.x版本中，Agent是构建在**LangGraph图结构**之上的，通过上文提到的**state**和**store**构建记忆系统。使用更简单、功能更统一。

* state：短期记忆对象，以 会话 为单位组织，包含当前会话的所有消息记录以及自定义信息。
* store：长期记忆对象， 跨会话持久化 的数据，通常需要结合向量数据库或外部存储实现。

#### 补充 1：v0.x 的 Memory 类在 v1.x 中怎么做（了解即可）

老教程里常见下面这些类。它们已经移到 `langchain-classic` 包中，新项目不再使用，看到时知道对应关系即可：

| v0.x 的类 | 做法 | v1.x 中的对应做法 |
|---|---|---|
| `ConversationBufferMemory` | 保存全部消息 | `checkpointer` 保存 state 中的 `messages` |
| `ConversationBufferWindowMemory` | 只保留最近 k 轮 | 在 `before_model` 中间件中裁剪消息 |
| `ConversationTokenBufferMemory` | 按 token 数保留最近的消息 | `trim_messages(strategy="last", max_tokens=...)` |
| `ConversationSummaryMemory`、`ConversationSummaryBufferMemory` | 把较早的消息总结成摘要 | `SummarizationMiddleware`（第八章 02 笔记） |
| `VectorStoreRetrieverMemory` | 存入向量库，按相关性检索 | 配置了向量索引的 `store`，用 `store.search(..., query=...)` 语义检索 |
| `RunnableWithMessageHistory` + `ChatMessageHistory`（v0.2 / v0.3 的写法） | 按 `session_id` 保存消息历史 | `checkpointer` + `thread_id` |

v1.x 统一之后，短期记忆就是 state，由 checkpointer 保存。第八章用过的人工审批（`HumanInTheLoopMiddleware` 暂停后恢复执行）、`thread_limit` 的累计计数，依靠的也是同一套 checkpointer 机制。

#### 补充 2：store 不等于向量数据库

store 本质上是“**命名空间 + key → JSON 文档**”的存储，类似“文件夹 + 文件名 → 文件内容”：

```python
store.put(("users", "u_001"), "profile", {"name": "小明"})   # 写入
store.get(("users", "u_001"), "profile")                    # 按 key 读取
store.search(("users", "u_001"), query="饮食偏好")            # 检索：按语义检索需要配置向量索引
```

* 只按 key 读写时，用不到向量数据库。
* 创建 store 时配置了嵌入模型（`index` 参数），`search` 才能按语义相似度检索，这时才用到向量能力。
* `InMemoryStore` 存在内存里，用于学习；生产环境使用 `PostgresStore`（可配合 pgvector 做语义检索）等持久化实现。

## 1.4 学习重点

| 档位 | 内容 | 理由 |
|---|---|---|
| 重点掌握<br>（弄懂原理并动手练习） | ⭐ `checkpointer` + `thread_id` | 短期记忆的基础；第八章的人工审批、`thread_limit` 也依赖它 |
| | ⭐ 短期记忆的管理：裁剪、删除、摘要 | 长对话几乎一定会遇到，可以和第八章的 `SummarizationMiddleware` 对照着学 |
| | ⭐ store 的 namespace / key，以及 `put`、`get`、`search` | 长期记忆的基本操作 |
| | ⭐ 在工具、中间件中通过 `runtime` 读写 state 和 store | 只有在 Agent 运行过程中读写，记忆才真正起作用 |
| 熟练使用 | 自定义 state 字段（`state_schema`） | 短期记忆里除了消息还要存其他信息时使用 |
| | 用 `context_schema` 传入 `user_id` | 长期记忆按用户区分的前提 |
| 按需学习 | 语义检索（给 store 配置嵌入模型） | 记忆条目多、需要按相关性查找时 |
| | 生产环境的持久化（`PostgresSaver`、`PostgresStore` 等） | 程序重启后记忆还要保留时 |
| 了解即可 | 长期记忆的三种类型、写入时机 | 设计记忆系统时的思路，不对应具体 API |
| | v0.x 的 Memory 类 | 只在阅读老代码时需要 |

第七章 01 笔记 2.5.5 已经演示过 `checkpointer` 的基本用法，下一节 `02_short_memory` 在此基础上深入短期记忆。